In [2]:
!pip install transformers torch openpyxl tqdm Pillow -q

In [1]:
# ============================================================
# LT-EDI @ ACL 2026: THE FINAL ROBUST GATED FUSION SCRIPT
# Handles: .jpg/.jpeg/.png, "Homophobic" vs "Homophobia", & Drive
# ============================================================

import os, torch, pandas as pd
import torch.nn as nn
from PIL import Image
from transformers import AutoTokenizer, AutoModel, CLIPProcessor, CLIPVisionModel
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm import tqdm
from google.colab import drive

# --- 1. SETUP ---
drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/LTEDI_DATA"
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. UNIQUE GATED MODEL ---
class GatedMemeModel(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        self.text_model = AutoModel.from_pretrained("xlm-roberta-base")
        self.vision_model = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")

        self.text_proj = nn.Linear(768, 512)
        self.vis_proj = nn.Linear(768, 512)

        self.gate = nn.Sequential(nn.Linear(1024, 512), nn.Sigmoid())

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 3)
        )
        self.loss_fn = nn.CrossEntropyLoss(weight=weights) if weights is not None else nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        t_out = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        t_feat = self.text_proj(t_out.last_hidden_state[:, 0, :])

        v_out = self.vision_model(pixel_values=pixel_values)
        v_feat = self.vis_proj(v_out.pooler_output)

        combined = torch.cat([t_feat, v_feat], dim=-1)
        g = self.gate(combined)
        fused = g * t_feat + (1 - g) * v_feat

        logits = self.classifier(fused)
        loss = self.loss_fn(logits, labels) if labels is not None else None
        return logits, loss

# --- 3. ROBUST DATASET HANDLER ---
class MemeDataset(Dataset):
    def __init__(self, lang_root, tokenizer, processor, is_test=False):
        self.is_test = is_test
        self.tokenizer = tokenizer
        self.processor = processor
        self.img_dir = os.path.join(lang_root, "Test_images" if is_test else "Train_images")
        excel_name = "Test_without_labels.xlsx" if is_test else "Train_labels.xlsx"
        self.df = pd.read_excel(os.path.join(lang_root, excel_name))

        # Mapping to handle label variations
        self.label_map = {
            "Homophobia": 0, "Homophobic": 0,
            "Transphobia": 1, "Transphobic": 1,
            "Non-anti LGBT": 2, "Non-anti-LGBT": 2, "Non-Anti LGBT": 2
        }

    def __len__(self): return len(self.df)

    def find_image(self, img_id):
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            path = os.path.join(self.img_dir, f"{img_id}{ext}")
            if os.path.exists(path): return path
        return None

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = str(row['id'])
        img_path = self.find_image(img_id)

        if img_path:
            image = Image.open(img_path).convert("RGB")
        else:
            image = Image.new('RGB', (224, 224), color='white')

        text = str(row.get('text', ""))
        t_inputs = self.tokenizer(text, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
        v_inputs = self.processor(images=image, return_tensors="pt")

        item = {
            'input_ids': t_inputs['input_ids'].squeeze(0),
            'attention_mask': t_inputs['attention_mask'].squeeze(0),
            'pixel_values': v_inputs['pixel_values'].squeeze(0),
            'id': img_id
        }
        if not self.is_test:
            raw_label = str(row['label']).strip()
            item['labels'] = torch.tensor(self.label_map.get(raw_label, 2))
        return item

# --- 4. EXECUTION PIPELINE ---
def run_pipeline():
    tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    # Priority weighting for Task 2 Macro-F1
    weights = torch.tensor([1.5, 2.0, 1.0]).to(device)
    model = GatedMemeModel(weights=weights).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

    langs = ['english', 'hindi', 'chinese']
    train_sets = [MemeDataset(os.path.join(BASE_PATH, l), tokenizer, processor) for l in langs if os.path.exists(os.path.join(BASE_PATH, l))]
    train_loader = DataLoader(ConcatDataset(train_sets), batch_size=8, shuffle=True)

    print("🚀 Training starting...")
    model.train()
    for epoch in range(3):
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}"):
            optimizer.zero_grad()
            logits, loss = model(batch['input_ids'].to(device), batch['attention_mask'].to(device),
                                batch['pixel_values'].to(device), batch['labels'].to(device))
            loss.backward()
            optimizer.step()

    print("📂 Generating Submissions...")
    inv_map = {0: "Homophobia", 1: "Transphobia", 2: "Non-anti LGBT"}
    model.eval()
    for l in langs:
        path = os.path.join(BASE_PATH, l)
        if not os.path.exists(path): continue
        test_ds = MemeDataset(path, tokenizer, processor, is_test=True)
        results = []
        for batch in tqdm(DataLoader(test_ds, batch_size=1), desc=f"Predicting {l}"):
            with torch.no_grad():
                logits, _ = model(batch['input_ids'].to(device), batch['attention_mask'].to(device),
                                  batch['pixel_values'].to(device))
                pred = torch.argmax(logits, dim=1).item()
                results.append([batch['id'][0], inv_map[pred]])
        pd.DataFrame(results).to_csv(f"Individual_{l}.csv", index=False, header=False)
    print("✅ All done! Download Individual_*.csv from the file sidebar.")

if __name__ == "__main__":
    run_pipeline()

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

🚀 Training starting...


Epoch 0:  57%|█████▋    | 165/290 [06:16<04:50,  2.32s/it]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 2: 100%|██████████| 290/290 [01:47<00:00,  2.69it/s]


📂 Generating Submissions...


Predicting chinese: 100%|██████████| 239/239 [01:39<00:00,  2.40it/s]


✅ All done! Download Individual_*.csv from the file sidebar.
